# 01. Carga con COPY INTO (Catálogo y Sucursales)

Primera tarea del job. Catálogo llega como snapshots completos (un arreglo JSON por archivo) y Sucursales es un archivo de referencia único: pocas filas, pocos archivos y carga por lotes. COPY INTO es idempotente (no vuelve a cargar un archivo ya cargado), escribe en una tabla Delta común de `bronze` y el pipeline principal la lee como fuente.

Las tablas Bronze guardan todo como texto, más las columnas técnicas de auditoría: archivo de origen (`_metadata.file_name`), momento de ingesta, sistema de origen, lote y la corrida del job.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if not CATALOGO or not all(c.isalnum() or c == "_" for c in CATALOGO):
    raise ValueError("El catálogo solo admite letras, números y guion bajo")

In [ ]:
dbutils.widgets.text("job_run_id", "manual")
JOB_RUN_ID = dbutils.widgets.get("job_run_id").strip()
if not JOB_RUN_ID or not all(c.isalnum() or c in "_-" for c in JOB_RUN_ID):
    raise ValueError("job_run_id solo admite letras, números, guion y guion bajo")
RUTA_LANDING = f"/Volumes/{CATALOGO}/default/vol_landing"

In [ ]:
import os
import sys

RAIZ_BUNDLE = os.path.abspath(os.path.join(os.getcwd(), ".."))
if RAIZ_BUNDLE not in sys.path:
    sys.path.append(RAIZ_BUNDLE)

In [ ]:
from src.electrocasa.utils.esquemas import AUDITORIA_COPY_INTO, CATALOGO as DDL_CATALOGO, SUCURSALES as DDL_SUCURSALES

CARGAS = [
    {
        "tabla": f"{CATALOGO}.bronze.catalogo",
        "ddl": DDL_CATALOGO,
        "columnas": "producto_id, nombre_producto, categoria, marca, precio_lista, activo",
        "carpeta": f"{RUTA_LANDING}/catalogo/",
        "formato": "JSON",
        "opciones": "'multiLine' = 'true', 'primitivesAsString' = 'true'",
        "sistema": "erp_catalogo",
    },
    {
        "tabla": f"{CATALOGO}.bronze.sucursales",
        "ddl": DDL_SUCURSALES,
        "columnas": "sucursal_id, nombre_sucursal, distrito, ciudad, region, fecha_apertura",
        "carpeta": f"{RUTA_LANDING}/sucursales/",
        "formato": "CSV",
        "opciones": "'header' = 'true'",
        "sistema": "maestro_sucursales",
    },
]

In [ ]:
for carga in CARGAS:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {carga['tabla']} ({carga['ddl']}, {AUDITORIA_COPY_INTO}) "
              f"COMMENT 'Bronze cargado con COPY INTO; columnas de negocio como texto'")
    resultado = spark.sql(f"""
        COPY INTO {carga['tabla']}
        FROM (
          SELECT {carga['columnas']},
                 _metadata.file_name AS source_file,
                 current_timestamp() AS ingestion_timestamp,
                 '{carga['sistema']}' AS source_system,
                 substring_index(_metadata.file_name, '.', 1) AS batch_id,
                 '{JOB_RUN_ID}' AS job_run_id
          FROM '{carga['carpeta']}'
        )
        FILEFORMAT = {carga['formato']}
        FORMAT_OPTIONS ({carga['opciones']})
        COPY_OPTIONS ('mergeSchema' = 'false')
    """)
    print(carga["tabla"])
    display(resultado)

In [ ]:
for carga in CARGAS:
    display(spark.sql(f"SELECT batch_id, count(*) AS filas FROM {carga['tabla']} GROUP BY batch_id ORDER BY batch_id"))